Question 1

In [ ]:
# Modified Code For Question 1 - From sigmoid to ReLU

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F



from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

# 2. Device Configuration and Seed Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import random

seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

# 3. Load CIFAR-10 Dataset

#Basic Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

#Download Dataset
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

#Create DataLoaders
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


# 4. CNN Starter Code
import torch
import torch.nn as nn
import torch.nn.functional as F

class InitialCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.fc1 = nn.Linear(256 * 2 * 2, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
# Changing the Activaition function from sigmoid to relu 
  #      x = torch.sigmoid(self.conv1(x))
        x = torch.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)

   #     x = torch.sigmoid(self.conv2(x))
        x = torch.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

    #    x = torch.sigmoid(self.conv3(x))
        x = torch.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)

     #   x = torch.sigmoid(self.conv4(x))
        x = torch.relu(self.conv4(x))
        x = F.max_pool2d(x, 2)

        x = x.view(x.size(0), -1)

      #  x = torch.sigmoid(self.fc1(x))
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# 5. MLP Starter Code
class InitialMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten

        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x

# 6. Initialize Network
model = InitialCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# 7. Training Function
def train(model, loader):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    gradient_norms = {}

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        # Save gradient norms
        for name, param in model.named_parameters():

            if param.grad is not None:

                grad_norm = param.grad.norm().item()

                if name not in gradient_norms:
                    gradient_norms[name] = []

                gradient_norms[name].append(grad_norm)

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total

    avg_loss = running_loss / len(loader)

    return avg_loss, accuracy, gradient_norms

# 8. Testing Function
def test(model, loader):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total

    avg_loss = running_loss / len(loader)

    return avg_loss, accuracy

# 9. Main Training Loop
num_epochs = 15

train_losses = []
test_losses = []

train_accs = []
test_accs = []

all_epochs_grad_norms = [] # To store gradient norms for all epochs

for epoch in range(num_epochs):

    train_loss, train_acc, grad_norms = train(
        model,
        train_loader
    )

    all_epochs_grad_norms.append(grad_norms) # Store grad_norms for the current epoch

    test_loss, test_acc = test(
        model,
        test_loader
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1}/{num_epochs}")

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")

    print("-" * 40)

# 10. Train the MLP
mlp_model = InitialMLP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=0.001
)

# 11. Example image visualization code
classes = train_dataset.classes

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2,5, figsize=(10,5))

for i, ax in enumerate(axes.flat):

    img = images[i] / 2 + 0.5
    img = img.permute(1,2,0)

    ax.imshow(img)
    ax.set_title(classes[labels[i]])
    ax.axis('off')

plt.show()

In [ ]:
# Process and visualize gradient norms (For ReLU and Sigmoid)

# Get all unique layer names from the first epoch's gradient norms
# Filter out bias terms to focus on weight gradients, as they are usually more indicative.
layer_names = [name for name in all_epochs_grad_norms[0].keys() if 'weight' in name]

# Initialize a dictionary to store average gradient norms per layer per epoch
avg_grad_norms_per_layer = {name: [] for name in layer_names}

# Populate the dictionary
for epoch_grad_norms in all_epochs_grad_norms:
    for name in layer_names:
        if name in epoch_grad_norms and len(epoch_grad_norms[name]) > 0:
            # Calculate the average norm across all batches in the epoch
            avg_norm = sum(epoch_grad_norms[name]) / len(epoch_grad_norms[name])
            avg_grad_norms_per_layer[name].append(avg_norm);
        else:
            # Append NaN or 0 if no gradients were recorded for this layer in this epoch
            avg_grad_norms_per_layer[name].append(0.0) # Or np.nan

# Plotting the gradient norms
plt.figure(figsize=(12, 7))
epochs = range(1, len(all_epochs_grad_norms) + 1);

for name, norms in avg_grad_norms_per_layer.items():
    plt.plot(epochs, norms, label=name)

plt.xlabel('Epoch')
plt.ylabel('Average Gradient Norm')
# plt.title('Graph 1: Average Gradient Norm per Layer Across Epochs with Sigmoid Activation')
plt.title('Graph 1: Average Gradient Norm per Layer Across Epochs with ReLU Activation')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Accuracy and learning curves 


epochs = range(1, num_epochs + 1)

plt.figure(figsize=(12, 5))

# Plotting training and test loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
plt.plot(epochs, train_losses, label='Train Loss')
plt.plot(epochs, test_losses, label='Test Loss')
plt.title('Graph 3: Training and Test Loss with ReLU Activation')
# plt.title('Graph 3: Training and Test Loss with Sigmoid Activation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting training and test accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
plt.plot(epochs, train_accs, label='Train Accuracy')
plt.plot(epochs, test_accs, label='Test Accuracy')
plt.title('Graph 4: Training and Test Accuracy with ReLU Activation')
# plt.title('Graph 4: Training and Test Accuracy with Sigmoid Activation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
def compute_and_plot_confusion_matrix(evaluated_model, data_loader, dynamic_device, plot_title="Confusion Matrix"):
    """
    Evaluates a trained model on a given dataset loader, computes a normalized
    confusion matrix, and visualizes it as a percentage heatmap.
    """
    # Set the network to evaluation mode (freezes dropout and batch normalization states)
    evaluated_model.eval()

    collected_predictions = []
    collected_ground_truths = []

    # Disable gradient tracking to optimize execution speed and VRAM overhead
    with torch.no_grad():
        for input_images, ground_truth_labels in data_loader:
            # Transfer tensor data to target compute hardware (GPU or CPU)
            input_images = input_images.to(dynamic_device)

            # Forward execution pass: map inputs to raw logit outputs
            logit_outputs = evaluated_model(input_images)

            # Extract predicted labels by capturing the highest logit response index
            _, batch_predictions = logit_outputs.max(1)

            # Store predictions and labels as standard numpy arrays on the host CPU
            collected_predictions.extend(batch_predictions.cpu().numpy())
            collected_ground_truths.extend(ground_truth_labels.numpy())

    # Explicit target labels mapped directly to official CIFAR-10 classification indices
    cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                       'dog', 'frog', 'horse', 'ship', 'truck']

    # Construct the underlying sample frequency confusion matrix
    raw_matrix = confusion_matrix(collected_ground_truths, collected_predictions)

    # Normalize values row-wise to convert raw sample frequencies into definitive percentages
    normalized_matrix = raw_matrix.astype('float') / raw_matrix.sum(axis=1)[:, np.newaxis] * 100

    # Initialize a clean canvas for heatmap rendering
    plt.figure(figsize=(10, 8))

    # Plot the normalized matrix using seaborn's annotated color grid
    sns.heatmap(normalized_matrix, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=cifar10_classes, yticklabels=cifar10_classes)

    # Apply standard academic labels and format sizes
    plt.title(plot_title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label (Ground Truth)', fontsize=12)
    plt.xlabel('Predicted Label (Model Classification)', fontsize=12)

    # Adjust geometric fit and draw matrix canvas
    plt.tight_layout()
    plt.show()

# To evaluate and plot the CNN (ReLU OR Sigmoid) model run:
compute_and_plot_confusion_matrix(
    model,
    test_loader,
    device,
    plot_title="Confusion Matrix 2: CNN (ReLU) Normalized Confusion Matrix")
   #  plot_title="Confusion Matrix 1: CNN (Sigmoid) Normalized Confusion Matrix")  # printing if using sigmoid